In [1]:
# Add repo root to Python path
import sys
from pathlib import Path

repo_root = Path.cwd().parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print(f"✅ Added to path: {repo_root}")

✅ Added to path: e:\self-evolving-dce_repo_bundle


## Import modules

In [2]:
from src.chunking.splitter import DocumentChunker, estimate_tokens
from src.ingestion.loaders import DataLoader
from src.config import Config

## Load clean documents

In [3]:
# Load documents from previous step
clean_docs_path = Config.MVP_KB_OUTPUTS / "mdn_learn_docs_clean.jsonl"
docs = DataLoader.load_jsonl(clean_docs_path)

print(f"📄 Loaded clean documents: {len(docs)}")
print(f"Sample document keys: {list(docs[0].keys())}")

📄 Loaded clean documents: 238
Sample document keys: ['doc_id', 'repo_rel_path', 'source_url_guess', 'title', 'text', 'meta']


## Initialize chunker

In [4]:
# Create chunker with default settings
chunker = DocumentChunker(
    max_chunk_tokens=300,
    min_chunk_tokens=80
)

print(f"Chunker settings:")
print(f"  Max tokens: {chunker.max_chunk_tokens}")
print(f"  Min tokens: {chunker.min_chunk_tokens}")

Chunker settings:
  Max tokens: 300
  Min tokens: 80


## Test on a single document

In [5]:
# Test chunking on first document
sample_doc = docs[0]
sample_chunks = chunker.chunk_document(sample_doc)

print(f"\nDocument: {sample_doc['title']}")
print(f"Chunks created: {len(sample_chunks)}")
print(f"\nFirst chunk:")
print(f"  ID: {sample_chunks[0]['chunk_id']}")
print(f"  Heading: {sample_chunks[0]['heading']}")
print(f"  Tokens: {sample_chunks[0]['tokens']}")
print(f"  Text preview: {sample_chunks[0]['text'][:200]}...")


Document: accessibility troubleshooting
Chunks created: 3

First chunk:
  ID: chunk_0
  Heading: Starting point
  Tokens: 197
  Text preview: To start this challenge, you should go and grab the [ZIP containing the files that comprise the example](https://raw.githubusercontent.com/mdn/learning-area/main/accessibility/assessment-start/assessm...


## Chunk all documents

In [6]:
# Process all documents
all_chunks = chunker.chunk_documents(docs)

print(f"\n✅ Chunking complete!")
print(f"Total chunks created: {len(all_chunks)}")


✅ Chunking complete!
Total chunks created: 2047


## Analyze chunk statistics

In [7]:
# Get statistics
stats = chunker.get_stats(all_chunks)

print("\n📊 Chunk Statistics:")
for key, value in stats.items():
    if isinstance(value, float):
        print(f"  {key}: {value:.2f}")
    else:
        print(f"  {key}: {value}")


📊 Chunk Statistics:
  total_chunks: 2047
  avg_tokens: 254.09
  min_tokens: 80
  max_tokens: 400
  total_tokens: 520121
  unique_documents: 232
  chunks_per_doc: 8.82


## Save chunks to JSONL

In [8]:
# Save to output file
chunks_output_path = Config.MVP_KB_OUTPUTS / "mdn_chunks.jsonl"
DataLoader.save_jsonl(all_chunks, chunks_output_path)

print(f"✅ Saved chunks to: {chunks_output_path}")
print(f"Total chunks: {len(all_chunks)}")

✅ Saved chunks to: e:\self-evolving-dce_repo_bundle\notebooks\mvp_kb_outputs\mdn_chunks.jsonl
Total chunks: 2047


## Inspect sample chunks (optional)

In [9]:
import pandas as pd
import random

# Create DataFrame for analysis
df = pd.DataFrame(all_chunks)

# Show random sample
print("\nRandom sample of chunks:")
display(df.sample(min(5, len(df))))

# Token distribution
print("\nToken distribution:")
print(df['tokens'].describe())


Random sample of chunks:


,chunk_id,doc_id,title,heading,heading_level,text,tokens,source_path,is_split,split_index
679,chunk_679,3656584b148d7ea6,vue first component,Vue's data object,2.0,If you change the value of the `label` prop pa...,400,files/en-us/learn_web_development/core/framewo...,True,0.0
1214,chunk_1214,e935e26a4c6c812f,lists,Marking up an unordered list,3.0,"To give you some practice, we want you to have...",110,files/en-us/learn_web_development/core/structu...,NaN,NaN
881,chunk_881,af7ddd9e36f12aa3,json,Converting between objects and text,2.0,The above example was simple in terms of acces...,400,files/en-us/learn_web_development/core/scripti...,True,0.0
801,chunk_801,44b679a2a2a021a6,debugging javascript,Summary,2.0,So that's JavaScript debugging and error handl...,84,files/en-us/learn_web_development/core/scripti...,NaN,NaN
976,chunk_976,155ba1c5a1de12e2,loops,Loops 3,2.0,"In this final task, you will test every number...",333,files/en-us/learn_web_development/core/scripti...,True,0.0



Token distribution:
count    2047.000000
mean      254.089399
std       110.923650
min        80.000000
25%       154.000000
50%       242.000000
75%       383.000000
max       400.000000
Name: tokens, dtype: float64


---

## Summary

**What this demo does:**
1. Loads cleaned documents from Step 01
2. Splits by markdown headings
3. Applies token limits (300 max, 80 min)
4. Creates retrieval-ready chunks
5. Saves to JSONL for embedding step

**Next step:** Embedding & Retrieval (see `demo_03_embedding_retrieval.ipynb`)